# Chart 2 — `SUBNET_TXN_RATE` + `VELOCITY_BURST` on one timeline

Two rules that both reject transactions, but **for different reasons**:

- **`SUBNET_TXN_RATE`** (`> 20 txn requests / 5s window` from any `/24` subnet) — catches a sudden burst of transactions arriving from a single subnet, regardless of which account each one is for. Credential-stuffing shape.
- **`VELOCITY_BURST`** (`> 5 txns / 30s rolling window for one account`) — catches a single account issuing too many transactions in a short time, regardless of which subnet they come from. Account-takeover shape.

`ProcessTransaction` evaluates both rules on every transaction. The order inside the SP is `HIGH_SPEND` → `VELOCITY_BURST` → subnet rules; the **first** rule that fires wins the `rule_name` field. That matters when both are simultaneously tripped (see Scenario C).

---

## The three scenarios on this timeline

- **Scenario A — around 00:00:40** — `SUBNET_TXN_RATE` fires **alone**. The subnet receives a sudden spike of 22 txns in a single 5-second window across several accounts; no single account is bursting (max per-account rolling-30s count = 3). Rejected txns carry `rule_name='SUBNET_TXN_RATE'`.
- **Scenario B — around 00:01:10** — `VELOCITY_BURST` fires **alone**. Subnet txn volume is normal (4 txns in that 5-second window), but one account — **Eve Wilson** — has issued 7 transactions in the last 30 seconds. Rejected txns are Eve's and carry `rule_name='VELOCITY_BURST'`.
- **Scenario C — around 00:01:30** — **both rules fire in the same window**. The subnet is seeing 25 txns in 5 seconds (txn-rate exceeded), AND Eve is bursting again (7 txns in 30 seconds rolling, velocity exceeded). The window's rejections are a mix:
  - Txns from Eve's account → `VELOCITY_BURST` (the SP evaluates velocity before the subnet rules, so her account hits that one first).
  - Txns from the other bursting accounts → `SUBNET_TXN_RATE` (velocity doesn't fire for them, but the subnet-wide count does).

The takeaway: a single transaction can only carry one rejection reason, but **inside a single 5-second window multiple rules can be firing on different transactions in parallel**. The third scenario makes that concrete.

## Color scheme (consistent with Chart 1)

- 🥣 **Cyan family** = subnet-wide txn-request count side (light = normal, dark = exceeded).
- 🟠 **Amber family** = per-account rolling-30s count side (light = normal, dark = exceeded).
- 🟢 **Green** = ACCEPTED (solid fill).
- ⚪ **Grey with colored ring** = REJECTED; ring color matches the signal family that caused the rejection (dark cyan ring = `SUBNET_TXN_RATE`, dark amber ring = `VELOCITY_BURST`).

---

**This notebook uses INLINE FAKE DATA to iterate on the chart design.** Live BQ queries are shown as commented `# BQ VARIANT:` blocks in each query cell.

In [1]:
# !pip install pandas plotly  # google-cloud-bigquery db-dtypes needed later for BQ swap

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from datetime import datetime, timedelta, timezone

pio.renderers.default = "plotly_mimetype+notebook_connected"

# ── Configuration ───────────────────────────────────────────────
TARGET_SUBNET            = "78.46.220.0"   # /24 subnet in focus
TXN_RULE_THRESHOLD       = 20              # SUBNET_TXN_RATE:  >N txn requests per 5s per subnet → reject
VELOCITY_RULE_THRESHOLD  = 5               # VELOCITY_BURST:   >N txns per 30s rolling per account → reject
WINDOW_SECONDS           = 5

T0 = datetime(2026, 10, 3, 14, 0, 0, tzinfo=timezone.utc)

# ─── Live BigQuery toggle ───────────────────────────────────────────────────
USE_BQ = False   # True = live BigQuery; False = synthetic (offline chart iteration)

BQ_PROJECT = "voltdb-operator"
BQ_DATASET = "beam_threat_detection"

if USE_BQ:
    from google.cloud import bigquery
    client = bigquery.Client(project=BQ_PROJECT)

    # Rebase BQ timestamps onto the T0 anchor so the chart x-axis layout stays
    # identical to the synthetic version (which runs a 2-minute window starting
    # at T0). We shift by `T0 - MIN(page_hits_raw.timestamp)`.
    _probe = client.query(f"""
        SELECT MIN(timestamp) AS t_min
        FROM `{BQ_PROJECT}.{BQ_DATASET}.page_hits_raw`
    """).to_dataframe()
    _data_t_min = _probe["t_min"].iloc[0]
    if _data_t_min is None:
        raise RuntimeError("page_hits_raw is empty; cannot compute T0 shift")
    _data_t_min = _data_t_min.to_pydatetime().replace(tzinfo=timezone.utc)
    _T0_SHIFT = T0 - _data_t_min   # add this to every BQ timestamp
    print(f"BQ data starts at {_data_t_min.isoformat()}; shifting by {_T0_SHIFT} to anchor at T0={T0.isoformat()}")

## Fake Query 1 — subnet txn-request count per 5s window on `203.0.113.0/24`

Mirrors the BQ query that goes here:

```sql
SELECT TIMESTAMP_SECONDS(DIV(UNIX_SECONDS(txn_time), 5) * 5) AS window_start,
       COUNT(*) AS txn_count
FROM `voltdb-operator.beam_threat_detection.transactions`
WHERE source_ip LIKE '203.0.113.%'
  AND txn_time BETWEEN @start AND @end
GROUP BY window_start
ORDER BY window_start
```

Hand-crafted to tell the three-scenario story:
- Around 00:00:40 — 22 txns → **Scenario A** (subnet-rate burst, no account bursting)
- Around 00:01:10 — 4 txns → **Scenario B** (quiet subnet, one account bursting)
- Around 00:01:30 — 25 txns → **Scenario C** (both bursting)

In [3]:
if USE_BQ:
    _subnet_prefix = ".".join(TARGET_SUBNET.split(".")[:3])
    df_txn_windows = client.query(f"""
        SELECT TIMESTAMP_SECONDS(DIV(UNIX_SECONDS(txn_time), 5) * 5) AS window_start,
               COUNT(*) AS txn_count
        FROM `{BQ_PROJECT}.{BQ_DATASET}.transactions`
        WHERE source_ip LIKE "{_subnet_prefix}.%"
        GROUP BY window_start
        ORDER BY window_start
    """).to_dataframe()
    if len(df_txn_windows) == 0:
        # No txns from this subnet — synthesise one row at T0 so downstream code
        # doesn't blow up on empty frames.
        df_txn_windows = pd.DataFrame({"window_start": [T0], "txn_count": [0]})
    else:
        df_txn_windows["window_start"] = df_txn_windows["window_start"] + _T0_SHIFT
    df_txn_windows["txn_exceeded"] = df_txn_windows["txn_count"] > TXN_RULE_THRESHOLD
    print(f"{len(df_txn_windows)} windows; {df_txn_windows['txn_exceeded'].sum()} windows exceed TXN_RULE_THRESHOLD ({TXN_RULE_THRESHOLD})")
else:
    txn_counts = [
        1, 2, 0, 1, 0, 2,          # w0-5   baseline
        1, 2,                      # w6-7   baseline
        22,                        # w8   SCENARIO A (SUBNET_TXN_RATE alone) — above threshold
        1, 2, 0,                   # w9-11  baseline recovery
        2, 1, 3,                   # w12-13-14 baseline (w14 has Eve-only burst below)
        4,                         # Wait: w14 placement — see below. (Keeping txn_counts simple)
        1, 2, 1,                   # w15-17 baseline
        25,                        # w18  SCENARIO C (BOTH fire) — above threshold
        1, 2, 1, 2, 1,             # w19-23 baseline
    ]
    # The quick fix: txn_counts above has 25 entries not 24 — reduce.
    txn_counts = [
        1, 2, 0, 1, 0, 2,          # w0-5   baseline
        1, 2,                      # w6-7   baseline (quiet before Scenario A)
        22,                        # w8    SCENARIO A — SUBNET_TXN_RATE fires alone
        1, 2, 0,                   # w9-11  recovery
        2, 1,                      # w12-13 baseline
        4,                         # w14   SCENARIO B window — subnet is QUIET (velocity-only story)
        1, 2, 1,                   # w15-17 baseline
        25,                        # w18   SCENARIO C — BOTH rules fire
        1, 2, 1, 2, 1,             # w19-23 recovery
    ]
    assert len(txn_counts) == 24, f"expected 24 windows, got {len(txn_counts)}"

    df_txn_windows = pd.DataFrame({
        "window_start": [T0 + timedelta(seconds=5 * i) for i in range(len(txn_counts))],
        "txn_count":    txn_counts,
    })
    df_txn_windows["txn_exceeded"] = df_txn_windows["txn_count"] > TXN_RULE_THRESHOLD
    print(f"{len(df_txn_windows)} windows; {df_txn_windows['txn_exceeded'].sum()} windows exceed TXN_RULE_THRESHOLD ({TXN_RULE_THRESHOLD})")
    df_txn_windows[df_txn_windows['txn_exceeded'] | (df_txn_windows['txn_count'] >= 4)].head(10)


11 windows; 0 windows exceed TXN_RULE_THRESHOLD (20)


/Users/mpopova/Marina/VoltProjects/voltdb-examples/beam-threat-detection/notebooks/.venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## Fake Query 2 — max per-account 30s-rolling txn count per window

Mirrors a BQ query that computes, for each 5s window, the MAX per-account rolling-30s count across all accounts active on this subnet:

```sql
WITH per_account AS (
  SELECT TIMESTAMP_SECONDS(DIV(UNIX_SECONDS(txn_time), 5) * 5) AS window_start,
         account_id,
         COUNT(*) OVER (
           PARTITION BY account_id
           ORDER BY UNIX_SECONDS(txn_time)
           RANGE BETWEEN 30 PRECEDING AND CURRENT ROW
         ) AS rolling_30s_count
  FROM `voltdb-operator.beam_threat_detection.transactions`
  WHERE source_ip LIKE '203.0.113.%'
    AND txn_time BETWEEN @start AND @end
)
SELECT window_start, MAX(rolling_30s_count) AS max_per_account_count_30s
FROM per_account GROUP BY window_start ORDER BY window_start
```

Hand-crafted:
- Around 00:00:40 — max per-account count = 3 (the subnet-rate burst spreads across accounts; no single account bursts)
- Around 00:01:10 — max per-account count = 7 (Eve bursts → **Scenario B**, VELOCITY_BURST fires alone)
- Around 00:01:30 — max per-account count = 7 (Eve bursts again → **Scenario C**, velocity + subnet both fire)

In [4]:
if USE_BQ:
    # Max per-account 30s-rolling txn count, bucketed into 5s windows on burst subnet.
    _subnet_prefix = ".".join(TARGET_SUBNET.split(".")[:3])
    df_velocity_windows = client.query(f"""
        WITH per_txn AS (
            SELECT txn_time, account_id,
                   COUNT(*) OVER (
                       PARTITION BY account_id
                       ORDER BY UNIX_SECONDS(txn_time)
                       RANGE BETWEEN 30 PRECEDING AND CURRENT ROW
                   ) AS rolling_30s_count
            FROM `{BQ_PROJECT}.{BQ_DATASET}.transactions`
            WHERE source_ip LIKE "{_subnet_prefix}.%"
        )
        SELECT TIMESTAMP_SECONDS(DIV(UNIX_SECONDS(txn_time), 5) * 5) AS window_start,
               MAX(rolling_30s_count) AS velocity_count
        FROM per_txn
        GROUP BY window_start
        ORDER BY window_start
    """).to_dataframe()
    if len(df_velocity_windows) == 0:
        df_velocity_windows = pd.DataFrame({"window_start": [T0], "velocity_count": [0]})
    else:
        df_velocity_windows["window_start"] = df_velocity_windows["window_start"] + _T0_SHIFT
    df_velocity_windows["velocity_exceeded"] = df_velocity_windows["velocity_count"] > VELOCITY_RULE_THRESHOLD
    print(f"{len(df_velocity_windows)} windows; "
          f"{df_velocity_windows['velocity_exceeded'].sum()} windows exceed VELOCITY_RULE_THRESHOLD ({VELOCITY_RULE_THRESHOLD})")
else:
    velocity_counts = [
        0, 0, 0, 0, 0, 0,          # w0-5   baseline
        1, 2,                      # w6-7   mild activity
        3,                         # w8    SCENARIO A — counts spread across accounts, no single-account burst
        2, 1, 0,                   # w9-11  recovery
        1, 2,                      # w12-13 Eve ramping up
        7,                         # w14   SCENARIO B — VELOCITY_BURST fires (Eve has 7 in 30s)
        4, 2, 1,                   # w15-17 Eve declining then resetting
        7,                         # w18   SCENARIO C — Eve bursts again, same window as subnet burst
        4, 2, 1, 0, 0,             # w19-23 declining
    ]
    assert len(velocity_counts) == 24

    df_velocity_windows = pd.DataFrame({
        "window_start":    df_txn_windows["window_start"],
        "velocity_count":  velocity_counts,
    })
    df_velocity_windows["velocity_exceeded"] = df_velocity_windows["velocity_count"] > VELOCITY_RULE_THRESHOLD
    print(f"{len(df_velocity_windows)} windows; "
          f"{df_velocity_windows['velocity_exceeded'].sum()} windows exceed VELOCITY_RULE_THRESHOLD ({VELOCITY_RULE_THRESHOLD})")
    df_velocity_windows[df_velocity_windows['velocity_exceeded'] | (df_velocity_windows['velocity_count'] >= 3)].head(10)


11 windows; 2 windows exceed VELOCITY_RULE_THRESHOLD (5)


/Users/mpopova/Marina/VoltProjects/voltdb-examples/beam-threat-detection/notebooks/.venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## Fake Query 3 — individual transaction attempts on `203.0.113.0/24`

Hand-crafted dots chosen so each scenario reads correctly:

- **Scenario A** — 3 REJECTED by `SUBNET_TXN_RATE` (different accounts, no account bursting).
- **Scenario B** — 2 REJECTED by `VELOCITY_BURST` (Eve's 6th and 7th transactions in 30 seconds).
- **Scenario C** — 2 REJECTED by `SUBNET_TXN_RATE` (other accounts) **and** 2 REJECTED by `VELOCITY_BURST` (Eve), all in the same 5-second window.
- A handful of ACCEPTED dots in safe windows.

In [5]:
if USE_BQ:
    _subnet_prefix = ".".join(TARGET_SUBNET.split(".")[:3])
    df_txns = client.query(f"""
        SELECT txn_time, source_ip, CAST(amount AS FLOAT64) AS amount,
               accepted, rule_name, reason, account_name
        FROM `{BQ_PROJECT}.{BQ_DATASET}.transactions`
        WHERE source_ip LIKE "{_subnet_prefix}.%"
        ORDER BY txn_time
    """).to_dataframe()
    df_txns["txn_time"] = df_txns["txn_time"] + _T0_SHIFT
else:
    df_txns = pd.DataFrame([
        # --- 4 ACCEPTED in safe windows (both lines below their thresholds) ---
        {"txn_time": T0 + timedelta(seconds=3),   "source_ip": "78.46.220.42",  "amount":   87.50, "accepted": 1, "rule_name": "NONE",            "reason": "Accepted",                                                                 "account_name": "Alice Johnson"},
        {"txn_time": T0 + timedelta(seconds=17),  "source_ip": "78.46.220.61",  "amount":  249.99, "accepted": 1, "rule_name": "NONE",            "reason": "Accepted",                                                                 "account_name": "Charlie Brown"},
        {"txn_time": T0 + timedelta(seconds=102), "source_ip": "78.46.220.112", "amount":  175.00, "accepted": 1, "rule_name": "NONE",            "reason": "Accepted",                                                                 "account_name": "Bob Smith"},
        {"txn_time": T0 + timedelta(seconds=112), "source_ip": "78.46.220.54",  "amount":  119.00, "accepted": 1, "rule_name": "NONE",            "reason": "Accepted",                                                                 "account_name": "Alice Johnson"},

        # --- Scenario A: 3 REJECTED by SUBNET_TXN_RATE in w8 (40-45s) ---
        {"txn_time": T0 + timedelta(seconds=41),  "source_ip": "78.46.220.19",  "amount":  310.00, "accepted": 0, "rule_name": "SUBNET_TXN_RATE", "reason": "Too Many Transactions from Subnet (>20/5s): count 21 from IP 78.46.220.19",  "account_name": "Alice Johnson"},
        {"txn_time": T0 + timedelta(seconds=43),  "source_ip": "78.46.220.31",  "amount":  125.00, "accepted": 0, "rule_name": "SUBNET_TXN_RATE", "reason": "Too Many Transactions from Subnet (>20/5s): count 22 from IP 78.46.220.31",  "account_name": "Bob Smith"},
        {"txn_time": T0 + timedelta(seconds=44),  "source_ip": "78.46.220.77",  "amount":  440.00, "accepted": 0, "rule_name": "SUBNET_TXN_RATE", "reason": "Too Many Transactions from Subnet (>20/5s): count 22 from IP 78.46.220.77",  "account_name": "Charlie Brown"},

        # --- Scenario B: 2 REJECTED by VELOCITY_BURST in w14 (70-75s), both Eve ---
        {"txn_time": T0 + timedelta(seconds=71),  "source_ip": "78.46.220.9",   "amount":  215.00, "accepted": 0, "rule_name": "VELOCITY_BURST",  "reason": "Too Many Transactions in 30 Seconds (>5): count 6 for account 5",         "account_name": "Eve Wilson"},
        {"txn_time": T0 + timedelta(seconds=74),  "source_ip": "78.46.220.9",   "amount":  450.00, "accepted": 0, "rule_name": "VELOCITY_BURST",  "reason": "Too Many Transactions in 30 Seconds (>5): count 7 for account 5",         "account_name": "Eve Wilson"},

        # --- Scenario C: both rules in w18 (90-95s) ---
        #     Eve bursting again (VELOCITY_BURST) + other accounts flooding the subnet (SUBNET_TXN_RATE)
        {"txn_time": T0 + timedelta(seconds=91),  "source_ip": "78.46.220.9",   "amount":  310.00, "accepted": 0, "rule_name": "VELOCITY_BURST",  "reason": "Too Many Transactions in 30 Seconds (>5): count 6 for account 5",         "account_name": "Eve Wilson"},
        {"txn_time": T0 + timedelta(seconds=92),  "source_ip": "78.46.220.21",  "amount":  210.00, "accepted": 0, "rule_name": "SUBNET_TXN_RATE", "reason": "Too Many Transactions from Subnet (>20/5s): count 23 from IP 78.46.220.21",  "account_name": "Bob Smith"},
        {"txn_time": T0 + timedelta(seconds=93),  "source_ip": "78.46.220.9",   "amount":  425.00, "accepted": 0, "rule_name": "VELOCITY_BURST",  "reason": "Too Many Transactions in 30 Seconds (>5): count 7 for account 5",         "account_name": "Eve Wilson"},
        {"txn_time": T0 + timedelta(seconds=94),  "source_ip": "78.46.220.88",  "amount":  320.00, "accepted": 0, "rule_name": "SUBNET_TXN_RATE", "reason": "Too Many Transactions from Subnet (>20/5s): count 25 from IP 78.46.220.88",  "account_name": "Diana Prince"},
    ])
    df_txns


/Users/mpopova/Marina/VoltProjects/voltdb-examples/beam-threat-detection/notebooks/.venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## The chart

In [6]:
import random

# ── Color scheme (consistent with Chart 1) ───────────────────────────────
COLOR_TXN_LIGHT       = "#4dd0e1"
COLOR_TXN_DARK        = "#00838f"
COLOR_VELOCITY_LIGHT  = "#ffb74d"
COLOR_VELOCITY_DARK   = "#e65100"
COLOR_ACCEPT          = "#2e7d32"
COLOR_REJECT_FILL     = "#9e9e9e"

RULE_RING_COLOR = {
    "SUBNET_TXN_RATE": COLOR_TXN_DARK,
    "VELOCITY_BURST":  COLOR_VELOCITY_DARK,
}
DOT_Y_ACCEPTED = 2
DOT_Y_VELOCITY = [7, 8, 9]
DOT_Y_TXN_RATE = [22, 23, 24, 25]

# Single axis range (both counts share one 0-30 y).
AXIS_MIN = 0
AXIS_MAX = 30

def contiguous_true_ranges(df, flag_col, start_col, window_seconds=WINDOW_SECONDS):
    ranges = []
    start = None
    for i in range(len(df)):
        if df[flag_col].iloc[i] and start is None:
            start = df[start_col].iloc[i]
        elif not df[flag_col].iloc[i] and start is not None:
            end = df[start_col].iloc[i - 1] + timedelta(seconds=window_seconds)
            ranges.append((start, end))
            start = None
    if start is not None:
        end = df[start_col].iloc[-1] + timedelta(seconds=window_seconds)
        ranges.append((start, end))
    return ranges

def step_hv_series(df, x_col, y_col, window_seconds=WINDOW_SECONDS):
    xs = list(df[x_col]) + [df[x_col].iloc[-1] + timedelta(seconds=window_seconds)]
    ys = list(df[y_col]) + [df[y_col].iloc[-1]]
    return xs, ys

# ── Dot-placement jitter: nudge each dot off the rate line by ~6% of axis
# range, biased to whichever side has room. Keeps dots from drawing directly
# on top of the counter / threshold lines.
_rng = random.Random(42)   # deterministic
_JITTER_FRAC = 0.06

def _jitter_y(y, axis_min=AXIS_MIN, axis_max=AXIS_MAX, frac=_JITTER_FRAC):
    gap = (axis_max - axis_min) * frac
    y_clamped = max(axis_min, min(axis_max, y))
    space_above = axis_max - y_clamped
    space_below = y_clamped - axis_min
    if space_below < gap:
        return y_clamped + gap
    if space_above < gap:
        return y_clamped - gap
    return y_clamped + (gap if _rng.random() < 0.5 else -gap)

txn_bad_periods      = contiguous_true_ranges(df_txn_windows,      "txn_exceeded",      "window_start")
velocity_bad_periods = contiguous_true_ranges(df_velocity_windows, "velocity_exceeded", "window_start")

fig = go.Figure()

for x0, x1 in txn_bad_periods:
    fig.add_shape(type="rect", x0=x0, x1=x1, y0=0, y1=1, yref="y domain",
                  fillcolor=COLOR_TXN_DARK, opacity=0.12, line_width=0, layer="below")
    fig.add_annotation(x=x0, xanchor="left", y=1, yanchor="top", yref="y domain",
                       text="TXN RATE exceeded", showarrow=False,
                       font=dict(color=COLOR_TXN_DARK, size=12))
for x0, x1 in velocity_bad_periods:
    fig.add_shape(type="rect", x0=x0, x1=x1, y0=0, y1=1, yref="y domain",
                  fillcolor=COLOR_VELOCITY_DARK, opacity=0.14, line_width=0, layer="below")
    fig.add_annotation(x=x1, xanchor="right", y=0.88, yanchor="top", yref="y domain",
                       text="VELOCITY exceeded", showarrow=False,
                       font=dict(color=COLOR_VELOCITY_DARK, size=12))

tx, ty = step_hv_series(df_txn_windows, "window_start", "txn_count")
fig.add_trace(go.Scatter(x=tx, y=ty, mode="lines",
    line=dict(color=COLOR_TXN_LIGHT, width=2.5, shape="hv"),
    name="Subnet txn requests per 5s", legendgroup="txn", hoverinfo="skip"))
tc = [COLOR_TXN_DARK if b else COLOR_TXN_LIGHT for b in df_txn_windows["txn_exceeded"]]
fig.add_trace(go.Scatter(x=df_txn_windows["window_start"], y=df_txn_windows["txn_count"],
    mode="markers", marker=dict(size=8, color=tc, line=dict(color=COLOR_TXN_DARK, width=1)),
    name="Subnet txn requests per 5s", legendgroup="txn", showlegend=False,
    hovertemplate="<b>%{y:,.0f} txn requests</b> in 5s window starting %{x|%H:%M:%S}<extra></extra>"))

vx, vy = step_hv_series(df_velocity_windows, "window_start", "velocity_count")
fig.add_trace(go.Scatter(x=vx, y=vy, mode="lines",
    line=dict(color=COLOR_VELOCITY_LIGHT, width=2.5, shape="hv"),
    name="Max per-account rolling-30s count", legendgroup="velocity", hoverinfo="skip"))
vc = [COLOR_VELOCITY_DARK if b else COLOR_VELOCITY_LIGHT for b in df_velocity_windows["velocity_exceeded"]]
fig.add_trace(go.Scatter(x=df_velocity_windows["window_start"], y=df_velocity_windows["velocity_count"],
    mode="markers", marker=dict(size=8, color=vc, line=dict(color=COLOR_VELOCITY_DARK, width=1)),
    name="Max per-account rolling-30s count", legendgroup="velocity", showlegend=False,
    hovertemplate="<b>%{y:,.0f} txns</b> for one account in 30s rolling window ending at %{x|%H:%M:%S}<extra></extra>"))

x_start = df_txn_windows["window_start"].min()
x_end   = df_txn_windows["window_start"].max() + timedelta(seconds=WINDOW_SECONDS)

fig.add_shape(type="line", x0=x_start, x1=x_end, y0=TXN_RULE_THRESHOLD, y1=TXN_RULE_THRESHOLD,
              line=dict(color=COLOR_TXN_DARK, width=2, dash="dash"))
fig.add_annotation(x=x_end, y=TXN_RULE_THRESHOLD, xanchor="right", yanchor="bottom",
                   text=f"SUBNET_TXN_RATE threshold ({TXN_RULE_THRESHOLD} txns / 5s)",
                   showarrow=False, font=dict(color=COLOR_TXN_DARK, size=13))

fig.add_shape(type="line", x0=x_start, x1=x_end, y0=VELOCITY_RULE_THRESHOLD, y1=VELOCITY_RULE_THRESHOLD,
              line=dict(color=COLOR_VELOCITY_DARK, width=2, dash="dash"))
fig.add_annotation(x=x_end, y=VELOCITY_RULE_THRESHOLD, xanchor="right", yanchor="bottom",
                   text=f"VELOCITY_BURST threshold ({VELOCITY_RULE_THRESHOLD} txns / 30s per account)",
                   showarrow=False, font=dict(color=COLOR_VELOCITY_DARK, size=13))

df_txns_plot = df_txns.copy()
df_txns_plot["outcome"] = df_txns_plot["accepted"].map({1: "ACCEPTED", 0: "REJECTED"})

for (outcome, rule), grp in df_txns_plot.groupby(["outcome", "rule_name"]):
    if outcome == "ACCEPTED":
        base_ys = [DOT_Y_ACCEPTED] * len(grp)
        marker = dict(size=14, color=COLOR_ACCEPT, line=dict(color="white", width=2), symbol="circle")
        label = "Transaction — ACCEPTED"
    else:
        if rule == "VELOCITY_BURST":
            base_ys = DOT_Y_VELOCITY[:len(grp)]
        elif rule == "SUBNET_TXN_RATE":
            base_ys = DOT_Y_TXN_RATE[:len(grp)]
        else:
            base_ys = [DOT_Y_ACCEPTED] * len(grp)
        ring = RULE_RING_COLOR.get(rule, "#424242")
        marker = dict(size=16, color=COLOR_REJECT_FILL, line=dict(color=ring, width=3), symbol="circle")
        label = f"Transaction — REJECTED ({rule})"
    # Nudge each dot off whichever rate line it would sit on.
    ys = [_jitter_y(y) for y in base_ys]
    fig.add_trace(go.Scatter(
        x=grp["txn_time"], y=ys, mode="markers", name=label, marker=marker,
        customdata=grp[["source_ip", "amount", "rule_name", "reason", "account_name"]].values,
        hovertemplate=(
            f"<b>%{{customdata[0]}}</b> — <b>{outcome}</b><br>"
            "amount=$%{customdata[1]:,.2f}<br>account=%{customdata[4]}<br>"
            "rule_name=%{customdata[2]}<br>reason=%{customdata[3]}<extra></extra>"
        )))

fig.update_layout(
    title=dict(text=(
        "<b>How SUBNET_TXN_RATE and VELOCITY_BURST interact on "
        f"<span style='color:#e91e63'>{TARGET_SUBNET}/24</span></b><br>"
        "<span style='font-size:14px'><span style='color:#00838f'>Scenario A — subnet-rate alone</span> · "
        "<span style='color:#e65100'>Scenario B — velocity alone</span> · "
        "<b>Scenario C — both rules fire in the same window</b>, per-txn reject reason depends on which rule the SP matched first.</span>"
    ), x=0.02, xanchor="left", y=0.97, font=dict(size=16)),
    paper_bgcolor="white", plot_bgcolor="white",
    font=dict(family="Segoe UI, sans-serif", size=14),
    legend=dict(orientation="h", yanchor="top", y=-0.12, xanchor="center", x=0.5,
                font=dict(size=15)),
    height=600, margin=dict(l=70, r=70, t=100, b=110),
)
fig.update_xaxes(title_text="Time (UTC)", showgrid=True, gridcolor="#dadce0",
                 showline=True, linecolor="#9aa0a6", linewidth=1,
                 ticks="outside", tickcolor="#9aa0a6",
                 title_font=dict(size=14), tickfont=dict(size=12))
fig.update_yaxes(title_text="Transaction counts per 5s (subnet) / per 30s (per account)",
                 range=[AXIS_MIN, AXIS_MAX], showgrid=True, gridcolor="#dadce0",
                 showline=True, linecolor="#9aa0a6", linewidth=1,
                 ticks="outside", tickcolor="#9aa0a6",
                 tickmode="linear", tick0=0, dtick=5, tickformat="d",
                 title_font=dict(size=14), tickfont=dict(size=12))
fig.show()

# ── PNG export ──────────────────────────────────────────────────────────────
import os
_OUT_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'notebooks', 'images')
os.makedirs(_OUT_DIR, exist_ok=True)
_SUFFIX = 'live' if USE_BQ else 'synthetic'
_PNG = os.path.join(_OUT_DIR, f'chart2_txn_velocity_timeline.{_SUFFIX}.png')
fig.write_image(_PNG, width=1600, height=1000, scale=2)
print(f'wrote {_PNG}')

wrote /Users/mpopova/Marina/VoltProjects/voltdb-examples/beam-threat-detection/notebooks/images/chart2_txn_velocity_timeline.live.png
